In [25]:
# load libraries
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import yfinance as yf

from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch
from arch import arch_model

In [ ]:
# All 30 stocks according to https://www.fool.com/investing/stock-market/indexes/dow-jones/
DJIA_TICKERS = [
    "MMM", "AMZN", "AXP", "AMGN", "AAPL", "BA", "CAT", "CVX", 
    "CSCO", "KO", "DIS", "GS", "HD", "HON", "IBM", "JNJ", "JPM", 
    "MCD", "MRK", "MSFT", "NKE", "NVDA", "PG", "CRM", "SHW", "TRV", 
    "UNH", "VZ", "V", "WMT"
] 

START  = "2011-02-28" # IMPORTANT: Start at 02-28 so that we have returns starting on the 03-01
END    = "2024-03-24" 

# Pull data
raw = yf.download(
    DJIA_TICKERS,
    start=START,
    end=END,
    auto_adjust=True,
    progress=False
)

prices = raw["Close"].dropna(how="all")

simple_return = prices.pct_change() # create a new var for simple returns

prices.head()

prices_tidy = ( # get prices in a tidy format
    prices
    .stack()
    .rename("price")
    .reset_index()
    .rename(columns={"level_1": "ticker"})
)

returns_tidy = ( # get returns in a tidy format
    simple_return
    .stack()
    .rename("return")
    .reset_index()
    .rename(columns={"level_1": "ticker"})
)

data = prices_tidy.merge( # combine both returns and prices
    returns_tidy,
    on=["Date", "Ticker"],
    how="inner"
)[30:] # Drop the first 30 to ensure we start on march the first with the returns. 
       # If not youd start on Feb 28 with NaNs on the return.This goes back to the 
       # comment near the line of code: START  = "2011-02-28"

data

#data.to_csv("stocks.csv", index=False)

